In [ ]:
from collections import Counter, deque
from math import sqrt
import queue
import time
import psutil

# Parent Map Implementation
def bfs(initial_state):
    start_time = time.time()
    frontier = [initial_state]
    explore = set()
    parent_map = {}
    frontier_state = {initial_state: True}
    depth_map = {initial_state: 0}
    max_depth = 0
    
    while frontier:
        state = frontier.pop(0)
        explore.add(state)
        frontier_state.pop(state)

        if state == "012345678":
            return find_path(parent_map, initial_state, len(explore), max_depth, time.time() - start_time)
        else:
            for child in generate_children(state):
                if child not in frontier_state and child not in explore:
                    parent_map[child] = state
                    frontier_state[child] = True
                    frontier.append(child)
                    depth_map[child] = depth_map[state] + 1
                    if depth_map[child] > max_depth:
                        max_depth = depth_map[child]

    return []


def dfs(initial_state):
    start_time = time.time()
    frontier = deque()
    frontier.append(initial_state)
    frontier_state = {initial_state: True}
    explore = set()
    parent_map = {}
    depth_map = {initial_state: 0}
    max_depth = 0
    
    while frontier:
        state = frontier.pop()
        frontier_state.pop(state)
        explore.add(state)

        if state == "012345678":
            return find_path(parent_map, initial_state, len(explore), max_depth, time.time() - start_time)
        else:
            for child in reversed(generate_children(state)):
                if child not in frontier_state and child not in explore:
                    parent_map[child] = state
                    frontier_state[child] = True
                    frontier.append(child)
                    depth_map[child] = depth_map[state] + 1
                    if depth_map[child] > max_depth:
                        max_depth = depth_map[child]

    return []


def is_solvable(state):
    inv_count = 0
    empty_value = 0
    for i in range(0, 9):
        for j in range(i + 1, 9):
            if int(state[j]) != empty_value and int(state[i]) != empty_value and int(state[i]) > int(state[j]):
                inv_count += 1
    return inv_count % 2 == 0


def swap(s, i, j):
    lst = list(s)
    lst[i], lst[j] = lst[j], lst[i]
    return ''.join(lst)


def generate_children(state):
    state = str(state)
    children = []

    index = state.index("0")

    if index > 2:
        children.append(swap(state, index, index - 3))

    if index < 6:
        children.append((swap(state, index, index + 3)))

    if index % 3 > 0:
        children.append(swap(state, index, index - 1))

    if index % 3 < 2:
        children.append(swap(state, index, index + 1))

    return children


def print_path(game, cost, nodes_explored, depth, time_taken):
    game = game[::-1]
    for state in game:
        for i in range(9):
            if i == 2 or i == 5 or i == 8:
                print(state[i] + " ")
            else:
                print(state[i], end=" ")
        print("---------------------------------------------")
    print("Cost = " + str(cost))
    print("Depth = " + str(depth))
    print("Nodes Explored = " + str(nodes_explored))
    print("Running Time = " + str(time_taken))


def find_path(parent_map, initial_state, nodes_explored, depth, time_taken):
    current_state = "012345678"
    path = [current_state]
    cost = 0
    
    while current_state != initial_state:
        path.append(parent_map[current_state])
        current_state = parent_map.get(current_state)
        cost += 1
    
    print_path(path, cost, nodes_explored, depth, time_taken)


def A_star(initial_state, heuristic):
    start_time = time.time()
    frontier = queue.PriorityQueue()
    frontier.put([heuristic(initial_state), initial_state])
    explored = set()
    parent_map = {}
    depth_map = {initial_state: 0}
    max_depth = 0
    
    while not frontier.empty():
        state1 = frontier.get()
        prev_cost = state1[0]
        state = state1[1]
        explored.add(state)

        if state == "012345678":
            return find_path(parent_map, initial_state, len(explored), max_depth, time.time() - start_time)

        for child in generate_children(state):
            if child not in frontier.queue and child not in explored:
                parent_map[child] = state
                frontier.put([heuristic(child) + prev_cost + 1, child])
                depth_map[child] = depth_map[state] + 1
                if depth_map[child] > max_depth:
                    max_depth = depth_map[child]

    return []


def manhattan_distance(state, goal="012345678", col=3):
    h = 0
    for c in state:
        index = state.find(c)
        target = goal.find(c)
        x_index = index % col
        y_index = index // col
        x_target = target % col
        y_target = target // col
        h = h + abs(x_index - x_target) + abs(y_index - y_target)
    return h


def euclides_distance(state, goal="012345678", col=3):
    h = 0
    for c in state:
        index = state.find(c)
        target = goal.find(c)
        x_index = index % col
        y_index = index // col
        x_target = target % col
        y_target = target // col
        h = h + sqrt((x_index - x_target) ** 2 + (y_index - y_target) ** 2)
    return h


def solver(initial_state, algorithm, heuristic=None):
    solution = None
    
    if algorithm == 1:
        solution = bfs(initial_state)
    elif algorithm == 2:
        solution = dfs(initial_state)
    elif algorithm == 3:
        if heuristic == 1:
            solution = A_star(initial_state, manhattan_distance)
        elif heuristic == 2:
            solution = A_star(initial_state, euclides_distance)

    end_mem = psutil.virtual_memory().used
    return solution, end_mem


if __name__ == '__main__':
    print("Welcome to 8_puzzle solver")

    while True:
        while True:
            initial_state = input("Insert initial state: ")
            freq = Counter(initial_state)
            if len(initial_state) == len(freq) == 9 and initial_state.isdecimal():
                break
            print("Wrong initial state")

        print("Algorithms\n"
              "-----------\n"
              "[1] BFS\n"
              "[2] DFS\n"
              "[3] A*\n")
        while True:
            switch = int(input("=>"))
            if switch in range(1, 4):
                break
            print("Invalid choice")

        if not is_solvable(initial_state):
            print(initial_state + " Cannot be solved")
            continue

        if switch == 1 or switch == 2:
            solution = solver(initial_state, switch)
        elif switch == 3:
            while True:
                h = int(input("Choose heuristic:\n"
                              "-----------------\n"
                              "[1] Manhattan\n"
                              "[2] Euclidean\n"))
                if h in [1, 2]:
                    break
                print("Invalid choice")

            if h in [1, 2]:
                solution = solver(initial_state, switch, h)


Welcome to 8_puzzle solver
Insert initial state: 123405678
Algorithms
-----------
[1] BFS
[2] DFS
[3] A*

=>1
1 2 3 
4 0 5 
6 7 8 
---------------------------------------------
1 2 3 
0 4 5 
6 7 8 
---------------------------------------------
0 2 3 
1 4 5 
6 7 8 
---------------------------------------------
2 0 3 
1 4 5 
6 7 8 
---------------------------------------------
2 3 0 
1 4 5 
6 7 8 
---------------------------------------------
2 3 5 
1 4 0 
6 7 8 
---------------------------------------------
2 3 5 
1 0 4 
6 7 8 
---------------------------------------------
2 0 5 
1 3 4 
6 7 8 
---------------------------------------------
0 2 5 
1 3 4 
6 7 8 
---------------------------------------------
1 2 5 
0 3 4 
6 7 8 
---------------------------------------------
1 2 5 
3 0 4 
6 7 8 
---------------------------------------------
1 2 5 
3 4 0 
6 7 8 
---------------------------------------------
1 2 0 
3 4 5 
6 7 8 
---------------------------------------------
1 0 2 
3 4 5 
6 7 8